# Interference

> The medium a step's transmissions share: the power of the other senders at each receiver, half-duplex radios, and the SINR.

In [ ]:
#| default_exp interference

In [ ]:
#| hide
from nbdev.showdoc import *

The transmissions of a step happen at the same time, so they interfere. `Interference` gives each one the power the
other senders put at its receiver,

`SINR = P |h|² / (noise_power + Σ P' |h'|²)`,

where `P'` is another sender's power and `h'` the coefficient from it to the receiver, from the same fading. The
interference is treated as Gaussian noise. Then:

* **Half-duplex** (the default): a node that transmits in a step cannot receive in it. Its transmission is not
  `active` any more, and `half_duplex` says why.
* **One radio per node**: a sender's transmissions in a step don't interfere with each other, and it interferes
  with others at its strongest transmit power.
* **Power control** is open-loop: `ChannelInversion` targets the SNR, not the SINR.

`SINR` then gives the SINR at each subcarrier. Both are blocks (see `block`); `InterferenceBackend` (see `radio`)
puts `Interference` into a channel's pipeline, after the power control.

Sionna has no module for it: its SINR comes from the receiver (`PostEqualizationSINR` in `sionna.phy.ofdm`), with
the interference in the channel of every transmitter to every receiver (`StreamManagement`). Here it is summed from
the fading of each pair of nodes, which scales to many nodes.

In [ ]:
#| export
from __future__ import annotations

import math
from typing import Sequence, Tuple

import numpy as np

from comm_core.block import Block
from comm_core.core import Transmission
from comm_core.fading import Fading
from comm_core.topology import Link

In [ ]:
#| export
class Interference(Block):
    "The power the other senders put at each receiver, from the fading's coefficients between them; and half-duplex."

    outputs = ('interference', 'active', 'half_duplex')

    def __init__(
        self,
        fading: Fading, # Gives the interfering paths (the channel's fading)
        half_duplex: bool = True, # A node that transmits in a step cannot receive in it
    ):
        self.fading, self.half_duplex = fading, half_duplex

    def children(self) -> list:
        "None: the fading is the channel's, reset and updated there."
        return []

    def call(
        self,
        transmission: Sequence[Transmission], # The transmissions of the step
        power: np.ndarray, # Their transmit powers
        active: np.ndarray, # Whether each one is sent
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        "(interference, active, half_duplex): `active` without the transmissions whose receiver was transmitting (`half_duplex`)."
        emitted = {}                                   # sender -> its power on the air (its strongest transmission)
        for t, p, a in zip(transmission, power, active):
            if a:
                emitted[t.sender] = max(emitted.get(t.sender, 0.0), float(p))
        gains = {}                                     # |h|^2 of the interfering paths, once per step

        def gain(sender, receiver, message):
            if (sender, receiver) not in gains:
                path = Transmission(message=message, sender=sender, receiver=receiver, start_time=0.0)
                gains[sender, receiver] = abs(complex(self.fading.coefficient(path, Link(sender, receiver)))) ** 2
            return gains[sender, receiver]

        n = len(transmission)
        interference, deaf = np.zeros(n), np.zeros(n, dtype=bool)
        for k, t in enumerate(transmission):
            if active[k] and self.half_duplex and t.receiver in emitted:
                deaf[k] = True
                continue
            interference[k] = sum(p * gain(s, t.receiver, t.message) for s, p in emitted.items()
                                  if p > 0 and s not in (t.sender, t.receiver))
        return interference, np.asarray(active, bool) & ~deaf, deaf

In [ ]:
show_doc(Interference.call)

In [ ]:
#| export
class SINR(Block):
    "The SINR of each transmission at each subcarrier: P |h|² / (noise power + interference); infinite between nodes at the same place."

    outputs = ('sinr',)

    def call(
        self,
        h: np.ndarray, # h of each transmission at each subcarrier, (n, K)
        power: np.ndarray, # Their transmit powers, (n,)
        noise_power: float, # Noise power at the receivers
        interference: np.ndarray | None = None, # Power of the other transmissions at each receiver, (n,)
    ) -> np.ndarray:
        "Linear SINR, (n, K)."
        gain = np.abs(np.asarray(h)) ** 2
        noise = noise_power + (0.0 if interference is None else np.asarray(interference, float)[:, None])
        with np.errstate(divide='ignore', invalid='ignore', over='ignore'):
            sinr = np.asarray(power, float)[:, None] * gain / noise
        sinr[np.isinf(gain).any(1)] = math.inf
        return sinr

In [ ]:
show_doc(SINR.call)

### Tests

In [ ]:
from comm_core.core import Message
from comm_core.fading import RayTracedFading

In [ ]:
# a line of cells, |H|^2 = 1e-6 / d^3; a -> b, c -> d and b -> a at once
xs = np.arange(5)
d = np.abs(np.subtract.outer(xs, xs)).astype(float)
table = RayTracedFading(np.where(d > 0, 1e-3 / np.maximum(d, 1) ** 1.5, 0), [(x, 0) for x in xs],
                        node_positions={'a': (0, 0), 'b': (1, 0), 'c': (3, 0), 'd': (4, 0)})
step = [Transmission(Message(s, r, None), s, r, 0.0) for s, r in (('a', 'b'), ('c', 'd'), ('b', 'a'))]
power, active = np.ones(3), np.ones(3, dtype=bool)
interference, now_active, deaf = Interference(table)(step, power, active)
assert deaf.tolist() == [True, False, True] and now_active.tolist() == [False, True, False]   # a and b both transmit
assert abs(interference[1] - (1e-6 / 4 ** 3 + 1e-6 / 3 ** 3)) < 1e-18                          # a and b at d
# full-duplex: everyone hears, with the others' power; a sender does not interfere with itself
interference, now_active, deaf = Interference(table, half_duplex=False)(step, power, active)
assert not deaf.any() and abs(interference[0] - 1e-6 / 2 ** 3) < 1e-18                          # c at b
# not sent: no interference from it
interference, _, _ = Interference(table)(step[:2], power[:2], np.array([True, False]))
assert interference[0] == 0
# the SINR: P |h|^2 / (noise + interference), at each subcarrier; infinite between nodes at the same place
h = table(step, [Link(t.sender, t.receiver) for t in step])
i = np.array([0.0, 1e-7, 0.0])
sinr = SINR()(h, power, 1e-10, i)
assert sinr.shape == (3, 1) and abs(sinr[1, 0] / (1e-6 / (1e-10 + 1e-7)) - 1) < 1e-12
assert SINR()(np.array([[math.inf]]), np.array([0.0]), 1.0)[0, 0] == math.inf

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()